# Hackathon - Customer Purchase Prediction

We want to build a machine learning model to predict if a customer will buy an item or not.

### Step 1: Importing libraries

In [34]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

### Step 2: Loading train and test datasets

In [35]:
public_test=pd.read_csv("./week2_hackathon/public_test.csv")
private_test=pd.read_csv("./week2_hackathon/private_test.csv")
train=pd.read_csv("./week2_hackathon/train.csv")


### Step 3: Checking for missing values

In [36]:
public_test_missing=public_test.isnull().sum()
private_test_missing=private_test.isnull().sum()
train_missing=train.isnull().sum()

### Checking how the target values are distributed

In [ ]:
print(public_test_missing[:])
print("\n")
print(private_test_missing[:])
print(train_missing[:])


print("=== Target distribution (train.csv) ===")
print(train["Converted"].value_counts())
print("\nPercentage:")
print(train["Converted"].value_counts(normalize=True) * 100)
print("\n=== Target distribution (public_test.csv) ===")
print(public_test["Converted"].value_counts())
print("\nPercentage:")
print(public_test["Converted"].value_counts(normalize=True) * 100)

### Looking at the data stats

In [39]:
train.describe()

,User_ID,Age,Income,City_Tier,Pages_Viewed,Products_Viewed,Time_On_Site,Previous_Purchases,Discount_Seen,Browser_Version,Campaign_Code,Converted
count,10000.00000,8520.000000,9016.000000,10000.000000,10000.00000,10000.000000,8152.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,5000.50000,41.457746,69961.772797,1.933400,15.60850,15.219300,13.667241,2.972500,0.540700,12.473900,5511.162200,0.30870
std,2886.89568,13.770164,24790.673822,0.737712,8.62346,8.927563,19.438244,1.738691,0.498366,6.892856,2596.933802,0.46198
min,1.00000,18.000000,12000.000000,1.000000,1.00000,1.000000,0.800000,0.000000,0.000000,1.000000,1000.000000,0.00000
25%,2500.75000,29.000000,52294.644359,1.000000,8.00000,8.000000,7.640000,2.000000,0.000000,7.000000,3251.750000,0.00000
50%,5000.50000,41.000000,70171.613672,2.000000,16.00000,15.000000,11.145000,3.000000,1.000000,12.000000,5515.500000,0.00000
75%,7500.25000,53.000000,86907.747154,2.000000,23.00000,23.000000,15.630000,4.000000,1.000000,18.000000,7759.000000,1.00000
max,10000.00000,65.000000,161687.774167,3.000000,30.00000,37.000000,607.390000,12.000000,1.000000,24.000000,9998.000000,1.00000


### Step 4: Preprocessing the data
Splitting the train data into features (X) and target variable (y).

In [50]:
#Processing the given data....
#x-> the features
#y-> the result whether the customer bought it or not


x_train = train.drop(columns=["Converted"])
y_train = train["Converted"]


### Getting rid of text columns
We only keep numeric columns because our model cannot use text columns directly.

In [ ]:
#identifying text vs numeric data


numeric_cols=x_train.select_dtypes(include=["int64","float64"]).columns.tolist()
text_cols = x_train.select_dtypes(include=["object"]).columns.tolist()



#clearly we have 2 text columns the model cant recognize them and ignores them, for now we are removing those...

x_train=x_train[numeric_cols]
x_privatetest=private_test[numeric_cols]
x_publictest=public_test[numeric_cols]
print(x_train.loc[[0,1,2]])
print(x_privatetest.loc[[0,1,2]])
print(x_publictest.loc[[0,1,2]])


### Filling missing values
Filing null values with the median of each column.

In [ ]:
# Handling missing data, using median to fill them
imputer = SimpleImputer(strategy="median")
x_train_imputed = imputer.fit_transform(x_train)
x_public_imputed = imputer.transform(x_publictest)
x_private_imputed = imputer.transform(x_privatetest)

x_train_imputed = pd.DataFrame(x_train_imputed, columns=numeric_cols)
x_public_imputed = pd.DataFrame(x_public_imputed, columns=numeric_cols)
x_private_imputed = pd.DataFrame(x_private_imputed, columns=numeric_cols)



### Scaling the features
Let's scale everything using StandardScaler.

In [ ]:
#Scaling data 
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train_imputed)
#using the test datas scaling criteria to scale the tests
x_public_scaled = scaler.transform(x_public_imputed)
x_private_scaled = scaler.transform(x_private_imputed)
train_scaled_df = pd.DataFrame(x_train_scaled, columns=numeric_cols)
public_scaled_df = pd.DataFrame(x_public_scaled, columns=numeric_cols)
private_scaled_df = pd.DataFrame(x_private_scaled, columns=numeric_cols)
print(train_scaled_df.loc[[0,1,2]])
print(public_scaled_df.loc[[0,1,2]])
print(private_scaled_df.loc[[0,1,2]])


### Step 5: Training the Logistic Regression model

In [114]:

# Training the model
model = LogisticRegression(max_iter=1000,random_state=42)
print("Training the model on the provided training data\n")
model.fit(x_train_scaled,y_train)

coefficients = pd.DataFrame({
    'Feature': numeric_cols,
    'Coefficient': model.coef_[0]
})
coefficients = coefficients.sort_values('Coefficient', ascending=False)
print(coefficients)
print("\nIntercept:", model.intercept_[0])

Training the model on the provided training data

               Feature  Coefficient
4         Pages_Viewed     0.461289
5      Products_Viewed     0.315196
8        Discount_Seen     0.265165
7   Previous_Purchases     0.237425
6         Time_On_Site     0.189191
2               Income     0.107116
9      Browser_Version     0.002014
0              User_ID    -0.003912
1                  Age    -0.004016
10       Campaign_Code    -0.019135
3            City_Tier    -0.039349

Intercept: -0.9355519700827112


### Model accuracy on training data

In [116]:
# Training accuracy
accuracy = model.score(x_train_scaled,y_train)
print(f"The accuracy of the model trained on the basis of the provided data is {accuracy * 100}")

The accuracy of the model trained on the basis of the provided data is 70.97


### Step 6: Predictions on the public test set

In [119]:
y_public_prediction=model.predict(x_public_scaled)
print(y_public_prediction[0:20])

[0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 1 0 0 0 1]


### F1 score for public test

In [126]:
ypublic=public_test["Converted"]
f1= f1_score(ypublic,y_public_prediction)

print(f"F1 Score: {f1:.4f}")

F1 Score: 0.3550


### Step 7: Predictions on the private test set

In [ ]:
y_private_predictions=model.predict(x_private_scaled)

print(f"First 20 predictions:{y_private_predictions[0:20]}")

### Creating the submission.csv file

In [131]:

# Create submission.csv
print("Creating submission.csv...\n")

# Create DataFrame with User_ID and predictions
submission = pd.DataFrame({
    "User_ID": private_test["User_ID"],
    "Converted": y_private_predictions
})

print("First 10 rows of submission:")
print(submission.head(10))

print("\nSubmission shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Data types:", submission.dtypes.tolist())

# Verify it's binary (only 0s and 1s)
unique_values = submission["Converted"].unique()
print(f"\nUnique values in Converted: {sorted(unique_values)}")
print("Binary values (0 and 1) only!" if len(unique_values) <= 2 else "❌ ERROR: Non-binary values!")

# Save to CSV
submission.to_csv("submission.csv", index=False)

print("\n✅ submission.csv created successfully!")
print(f"File saved in your current directory")

Creating submission.csv...

First 10 rows of submission:
   User_ID  Converted
0   103001          0
1   103002          0
2   103003          0
3   103004          0
4   103005          0
5   103006          0
6   103007          0
7   103008          0
8   103009          0
9   103010          0

Submission shape: (3000, 2)
Columns: ['User_ID', 'Converted']
Data types: [dtype('int64'), dtype('int64')]

Unique values in Converted: [np.int64(0), np.int64(1)]
✓ Binary values (0 and 1) only!

✅ submission.csv created successfully!
File saved in your current directory
